# Coembedding

`rx.coembed` generates matched reactant and product conformers in one coordinate frame. Each endpoint retains its own bonds and declared stereochemistry. Pairs are sorted by combined score; `pair[0]` selects the lowest-score pair.

Both endpoints need the same atom inventory. Atom maps establish correspondence independently of atom order. Keep transferred hydrogens explicit and mapped. CXSMILES can also carry metal geometry and donor-slot annotations, as in the Ru example below.

The examples use the public API and inline (CX)SMILES. Install `xyzrender` in the notebook environment for the rotating GIF overlays; it is included in the development environment (`uv sync`). Run cells in order.


In [ ]:
import rxembed as rx
import xyzrender
from IPython.display import display


## Algorithm and settings

1. Match atoms and identify spectators and reacting atoms from connectivity, bond-order and declared stereo changes. `react_atoms` can add reacting pairs.
2. Build a numerical union with ZERO changing bonds. Average endpoint bounds when their windows overlap; use their interval hull when disjoint. Graph-derived CPCI (Custom Pairwise Coulombic Interactions) preferences bias the shared seed.
3. Copy the seed to the original endpoint graphs, apply declared stereo, and prepare each endpoint with at most `seed_relax_iters=100` native UFF steps per attempt. Zero skips preparation; rejected preparation restores the shared seed.
4. Alternate native UFF minimizations with positional coupling. Each endpoint receives up to `joint_max_iters=100` steps per cycle, for at most `max_cycles=100` cycles. All atoms can move.
5. Validate both endpoints. Keep accepted pairs and replace missing or rejected candidates with fresh endpoint seeds without generated CPCI. Return surviving pairs if recovery underfills, and raise if none survives. A failed joint result can retain a valid prepared pair.

The objective is

$$E_R+E_P+\frac{w_S}{|S|}\sum_{i\in S}\|r_i-p_i\|^2+\frac{w_R}{|R|}\sum_{i\in R}\|r_i-p_i\|^2.$$

`alignment_weight=25` weights spectators and `react_weight=1.25` weights reacting atoms. Empty selections contribute zero. These penalties have no free movement allowance. Endpoint validity and a low score do not establish a transition state or a favorable reaction approach.

Use `seed_union=False` for endpoint seeding or `seed_cpci=False` to disable generated CPCI. Caller-supplied native RDKit parameters retain their own CPCI settings. `seed` reproduces the retry sequence. `verbose=True` reports preparation caps, actual joint iterations and cycles, stopping reasons, and scores; native preparation step counts are unavailable. Requests with `n <= 3` warn that more conformers are recommended.

Preserve the shared frame with `auto_align=False` in GIFs and `align=False` in XYZ exports. Each GIF below shows the lowest-score pair.


## 1-bromopentane and chloride

Chloride replaces bromide in `CCCCCBr.[Cl-]`. The mapped endpoints keep the carbon chain and both halides in correspondence. This quick start requests ten pairs and enables verbose diagnostics.


In [ ]:
pentyl_reactant_smiles = '[CH3:1][CH2:2][CH2:3][CH2:4][CH2:5][Br:6].[Cl-:7]'
pentyl_product_smiles = '[CH3:1][CH2:2][CH2:3][CH2:4][CH2:5][Cl:7].[Br-:6]'

pentyl_pair = rx.coembed(
    pentyl_reactant_smiles,
    pentyl_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    react_weight=10,
)
print(f"Accepted pairs: {len(pentyl_pair)}")
pentyl_reactant, pentyl_product = pentyl_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(pentyl_reactant),
    overlay=xyzrender.load(pentyl_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## SN2

Fluoride substitutes chloride on methyl chloride. The explicit mapped hydrogens keep the two endpoints on the same atom inventory.


In [ ]:
sn2_reactant_smiles = '[C:1]([H:2])([H:3])([H:4])[Cl:6].[F-:5]'
sn2_product_smiles = '[C:1]([H:2])([H:3])([H:4])[F:5].[Cl-:6]'

sn2_pair = rx.coembed(
    sn2_reactant_smiles,
    sn2_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=-1,
)
print(f"Accepted pairs: {len(sn2_pair)}")
sn2_reactant, sn2_product = sn2_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(sn2_reactant),
    overlay=xyzrender.load(sn2_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Ru carbonyl rearrangement (`ru-co`)

The Ru-bound carbamoyl becomes an O-bound formamide as a proton transfers from the pincer nitrogen. The CXSMILES donor-slot annotations preserve the selected octahedral coordination identity.


In [ ]:
ru_co_reactant_smiles = '[Ru+2:1]12(<-[P:2]([c:6]3[c:10]([H:11])[c:41]([H:42])[c:49]([H:50])[c:34]([H:35])[c:36]3[H:37])([c:13]3[c:16]([H:17])[c:53]([H:54])[c:59]([H:60])[c:57]([H:58])[c:39]3[H:40])[C:23]([H:24])([H:25])[C:27]([N:4]->1([C:7]([H:8])([H:9])[C:20]([P:3]->2([c:12]1[c:18]([H:19])[c:30]([H:31])[c:51]([H:52])[c:45]([H:46])[c:32]1[H:33])[c:26]1[c:14]([H:15])[c:43]([H:44])[c:55]([H:56])[c:61]([H:62])[c:47]1[H:48])([H:21])[H:22])[H:67])([H:28])[H:29])(<-[C-:38]#[O+:5])(<-[H-:63])<-[C-:65](=[O:64])[N:66]([H:68])[C:69]([H:70])([H:71])[H:72] |atomProp:0.atomNote.OCT:1.atomNote.s0:28.atomNote.s2:33.atomNote.s1:61.atomNote.s3:63.atomNote.s4:64.atomNote.s5|'
ru_co_product_smiles = '[Ru+2:1]12(<-[P:2]([c:6]3[c:10]([H:11])[c:41]([H:42])[c:49]([H:50])[c:34]([H:35])[c:36]3[H:37])([c:13]3[c:16]([H:17])[c:53]([H:54])[c:59]([H:60])[c:57]([H:58])[c:39]3[H:40])[C:23]([H:24])([H:25])[C:27]([N-:4]->1[C:7]([H:8])([H:9])[C:20]([P:3]->2([c:12]1[c:18]([H:19])[c:30]([H:31])[c:51]([H:52])[c:45]([H:46])[c:32]1[H:33])[c:26]1[c:14]([H:15])[c:43]([H:44])[c:55]([H:56])[c:61]([H:62])[c:47]1[H:48])([H:21])[H:22])([H:28])[H:29])(<-[C-:38]#[O+:5])(<-[H-:63])<-[O:64]=[C:65]([N:66]([H:68])[C:69]([H:70])([H:71])[H:72])[H:67] |atomProp:0.atomNote.OCT:1.atomNote.s0:28.atomNote.s2:33.atomNote.s1:60.atomNote.s3:62.atomNote.s4:63.atomNote.s5|'

ru_co_pair = rx.coembed(
    ru_co_reactant_smiles,
    ru_co_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=0,
)
print(f"Accepted pairs: {len(ru_co_pair)}")
ru_co_reactant, ru_co_product = ru_co_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(ru_co_reactant),
    overlay=xyzrender.load(ru_co_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Spiro fragmentation (`spiro-ts1`)

The C–N bond to morpholine breaks and the oxyanion becomes a carbonyl, retaining the declared spiro-framework stereochemistry. The example name labels an endpoint pair, not an optimized transition state.


In [ ]:
spiro_ts1_reactant_smiles = '[O:1]=[C:2]1[N:3]([c:4]2[c:5]([H:31])[c:6]([H:32])[c:7]([H:33])[c:8]([H:34])[c:9]2[H:35])[N:10]=[C:11]([C:12]([H:36])([H:37])[H:38])[C@@:13]12[O:14][C@:15]([N+:16]1([H:17])[C:18]([H:39])([H:40])[C:19]([H:41])([H:42])[O:20][C:21]([H:43])([H:44])[C:22]1([H:45])[H:46])([O-:23])[C@@:24]2([c:25]1[c:26]([H:48])[c:27]([H:49])[c:28]([H:50])[c:29]([H:51])[c:30]1[H:52])[H:47]'
spiro_ts1_product_smiles = '[N:16]1([H:17])[C:18]([H:39])([H:40])[C:19]([H:41])([H:42])[O:20][C:21]([H:43])([H:44])[C:22]1([H:45])[H:46].[O:1]=[C:2]1[N:3]([c:4]2[c:5]([H:31])[c:6]([H:32])[c:7]([H:33])[c:8]([H:34])[c:9]2[H:35])[N:10]=[C:11]([C:12]([H:36])([H:37])[H:38])[C@@:13]12[O:14][C:15](=[O:23])[C@@:24]2([c:25]1[c:26]([H:48])[c:27]([H:49])[c:28]([H:50])[c:29]([H:51])[c:30]1[H:52])[H:47]'

spiro_ts1_pair = rx.coembed(
    spiro_ts1_reactant_smiles,
    spiro_ts1_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=0,
)
print(f"Accepted pairs: {len(spiro_ts1_pair)}")
spiro_ts1_reactant, spiro_ts1_product = spiro_ts1_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(spiro_ts1_reactant),
    overlay=xyzrender.load(spiro_ts1_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))


## Diels–Alder cycloaddition

Butadiene and methyl acrylate form two C–C bonds and a cyclohexene ring. The mapped endpoints retain the same atoms as connectivity and bond orders change.
The product leaves its new stereocenter unspecified, so `stereo='free'` permits either configuration.


In [ ]:
diels_alder_reactant_smiles = '[C:1](=[C:2]([C:3](=[C:4]([H:15])[H:16])[H:14])[H:13])([H:11])[H:12].[C:5](=[C:6]([C:7](=[O:8])[O:9][C:10]([H:20])([H:21])[H:22])[H:19])([H:17])[H:18]'
diels_alder_product_smiles = '[C:1]1([H:11])([H:12])[C:2]([H:13])=[C:3]([H:14])[C:4]([H:15])([H:16])[C:5]([H:17])([H:18])[C:6]1([C:7](=[O:8])[O:9][C:10]([H:20])([H:21])[H:22])[H:19]'

diels_alder_pair = rx.coembed(
    diels_alder_reactant_smiles,
    diels_alder_product_smiles,
    n=10,
    seed=42,
    threads=1,
    verbose=True,
    charge=0,
    stereo="free",
)
print(f"Accepted pairs: {len(diels_alder_pair)}")
diels_alder_reactant, diels_alder_product = diels_alder_pair[0]
display(xyzrender.render_gif(
    xyzrender.load(diels_alder_reactant),
    overlay=xyzrender.load(diels_alder_product),
    auto_align=False,
    gif_rot=True,
    rot_frames=36,
    gif_fps=8,
    hy=True,
))
